In [1]:
!pip install transformers peft datasets accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00


In [2]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU available: True
GPU Name: Tesla T4


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("✅ TinyLlama loaded on GPU!")

Loading tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ TinyLlama loaded on GPU!


In [4]:
from peft import LoraConfig, get_peft_model

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Add LoRA adapters
model = get_peft_model(model, lora_config)

print("✅ LoRA adapters added!")

# Show trainable parameters
model.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU available: True
GPU Name: Tesla T4


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("✅ TinyLlama loaded on GPU!")

Loading tokenizer...


Loading model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ TinyLlama loaded on GPU!


In [3]:
from peft import LoraConfig, get_peft_model

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach LoRA adapters
model = get_peft_model(model, lora_config)

print("✅ LoRA adapters added successfully!")

# Show trainable parameters
model.print_trainable_parameters()

✅ LoRA adapters added successfully!
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [5]:
import json
from datasets import Dataset

with open("train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} training examples")

dataset = Dataset.from_list(data)

print(dataset)

Loaded 3 training examples
Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})


In [6]:
def tokenize_function(example):
    text = (
        "### Instruction:\n"
        + example["instruction"]
        + "\n\n### Response:\n"
        + example["response"]
    )

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = dataset.map(tokenize_function)

print("✅ Dataset tokenized successfully!")
print(tokenized_dataset)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ Dataset tokenized successfully!
Dataset({
    features: ['instruction', 'response', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./tinyllama-lora-results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

print("✅ Training arguments created!")

✅ Training arguments created!


In [8]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("✅ Trainer created successfully!")

✅ Trainer created successfully!


In [9]:
print("🚀 Starting LoRA fine-tuning...")

trainer.train()

print("✅ LoRA fine-tuning completed!")

🚀 Starting LoRA fine-tuning...


Step,Training Loss
1,13.442042
2,13.442043
3,13.442043


✅ LoRA fine-tuning completed!


In [10]:
model.save_pretrained("tinyllama-lora-adapter")
tokenizer.save_pretrained("tinyllama-lora-adapter")

print("✅ LoRA adapter saved successfully!")

✅ LoRA adapter saved successfully!


In [12]:
from peft import PeftModel

# Load LoRA adapter on top of TinyLlama
model = PeftModel.from_pretrained(
    model,
    "tinyllama-lora-adapter"
)

model.eval()

print("✅ Model ready for inference!")

✅ Model ready for inference!


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight', '

In [1]:
import torch

print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading base model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    "tinyllama-lora-adapter"
)

model.eval()

print("✅ Fine-tuned LoRA model ready!")

Loading base model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading LoRA adapter...
✅ Fine-tuned LoRA model ready!


In [3]:
prompt = """
### Instruction:
Explain machine learning in simple words.

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### Instruction:
Explain machine learning in simple words.

### Response:
Machine learning is a field of computer science that involves the use of algorithms and statistical models to learn from data. It is a powerful tool for predicting and making decisions based on large datasets. Machine learning algorithms are trained on large datasets to identify patterns


In [4]:
!pip install -q bitsandbytes

In [5]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.49.2


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)


print("✅ TinyLlama loaded with 4-bit quantization!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ TinyLlama loaded with 4-bit quantization!


In [7]:
from peft import LoraConfig, get_peft_model


qlora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


model_qlora = get_peft_model(
    model_qlora,
    qlora_config
)


print("✅ QLoRA adapters added successfully!")

model_qlora.print_trainable_parameters()

✅ QLoRA adapters added successfully!
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [8]:
from datasets import Dataset
import json

with open("train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset_qlora = Dataset.from_list(data)

print(f"Loaded {len(dataset_qlora)} training examples")
print(dataset_qlora)

Loaded 3 training examples
Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})


In [9]:
def tokenize_function(example):
    text = f"""
### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_qlora_dataset = dataset_qlora.map(
    tokenize_function
)

print("✅ QLoRA dataset tokenized successfully!")
print(tokenized_qlora_dataset)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

✅ QLoRA dataset tokenized successfully!
Dataset({
    features: ['instruction', 'response', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})


In [10]:
from transformers import TrainingArguments, Trainer


training_args_qlora = TrainingArguments(
    output_dir="./qlora_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)


trainer_qlora = Trainer(
    model=model_qlora,
    args=training_args_qlora,
    train_dataset=tokenized_qlora_dataset
)


print("✅ QLoRA Trainer created successfully!")

✅ QLoRA Trainer created successfully!


In [11]:
print("🚀 Starting QLoRA fine-tuning...")

trainer_qlora.train()

print("✅ QLoRA fine-tuning completed!")

🚀 Starting QLoRA fine-tuning...


Step,Training Loss
1,14.697608
2,15.324142
3,14.239720
4,14.239720
5,14.697608
6,15.012156
7,13.699894
8,14.725664
9,13.967148


✅ QLoRA fine-tuning completed!


In [12]:
model_qlora.save_pretrained("qlora_adapter")
tokenizer.save_pretrained("qlora_adapter")

print("✅ QLoRA adapter saved successfully!")

✅ QLoRA adapter saved successfully!


In [13]:
import torch
from peft import PeftModel

# Load base model again
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load QLoRA adapter
qlora_model = PeftModel.from_pretrained(
    base_model,
    "qlora_adapter"
)

qlora_model.eval()

print("✅ QLoRA model ready for inference!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ QLoRA model ready for inference!


In [5]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
